# Explorative Analyse: Arbeitsmarkt Regensburg (Jobcenter) — Teil 4: Fluktuation

Baut auf den in [`01_Exploration.ipynb`](01_Exploration.ipynb) exportierten `eckwerte_long.csv` auf — kein erneutes Einlesen der Excel-Rohdaten nötig. Während [`02_Analyse_SGBII.ipynb`](02_Analyse_SGBII.ipynb) den *Bestand* an Arbeitslosen über die Zeit betrachtet, geht es hier um die Dynamik dahinter: **Zugang** (neu arbeitslos gemeldet) und **Abgang** (nicht mehr arbeitslos gemeldet) im Rechtskreis SGB II, Januar 2025 – September 2026.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

PROCESSED_DIR = Path("..") / "Data" / "processed"

df_long = pd.read_csv(PROCESSED_DIR / "eckwerte_long.csv")
df_long["period"] = pd.PeriodIndex(df_long["period"], freq="M").to_timestamp()

kennzahlen = pd.read_csv(PROCESSED_DIR / "kennzahlen_sgb2.csv", index_col="period", parse_dates=["period"])
kennzahlen = kennzahlen.sort_index()

merkmale = ["Zugang (im Monat)", "Abgang (im Monat)", "Zugang (12-Monatssumme)", "Abgang (12-Monatssumme)"]
fluktuation = df_long[(df_long["kategorie"] == "Arbeitslose") & (df_long["merkmal"].isin(merkmale))]
fluktuation = fluktuation.pivot_table(index="period", columns="merkmal", values="sgb2").sort_index()
fluktuation.tail()

ABBILDUNGEN_DIR = Path("..") / "Docs" / "Abbildungen"


def export_abbildung(fig, dateiname, width, height, **layout):
    """Speichert eine Kopie der Grafik im weissen Layout der Projektdokumentation
    (Docs/Abbildungen, doppelte Aufloesung); die Anzeige im Notebook bleibt unveraendert.
    Optionale `layout`-Angaben (z. B. kuerzerer Titel) gelten nur fuer die exportierte Datei."""
    go.Figure(fig).update_layout(template="plotly_white", **layout).write_image(
        ABBILDUNGEN_DIR / dateiname, scale=2, width=width, height=height
    )

## 1. Zugang und Abgang im Monatsverlauf

**Zugang (im Monat)** = Personen, die im jeweiligen Monat neu arbeitslos gemeldet wurden. **Abgang (im Monat)** = Personen, die im jeweiligen Monat aus der Arbeitslosigkeit abgegangen sind (in Arbeit, Ausbildung, Krankheit, aus anderen Gründen — die Abgangsgründe selbst stehen erst im Blatt „2.3 Arbeitslosigkeit Abgang“ und sind hier nicht enthalten). Beide Werte beziehen sich auf den Rechtskreis SGB II.

In [2]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=fluktuation.index, y=fluktuation["Zugang (im Monat)"], name="Zugang",
    marker_color="#dc2626",
))
fig.add_trace(go.Bar(
    x=fluktuation.index, y=-fluktuation["Abgang (im Monat)"], name="Abgang",
    marker_color="#16a34a",
))
fig.update_layout(
    title="Zugang und Abgang aus Arbeitslosigkeit im Monatsverlauf (SGB II)",
    yaxis_title="Personen (Abgang als negativer Balken)",
    xaxis_title="Monat",
    barmode="relative",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.12),
    height=450,
)
export_abbildung(fig, "abb10_zugang_abgang.png", 1000, 480)
fig

Der monatliche Zugang schwankt zwischen 215 und 346 Personen (Ø 270), der Abgang zwischen 185 und 369 Personen (Ø 283) — im Mittel gehen also etwas mehr Personen ab als neu hinzukommen. Der Zugang erreichte im Juni 2026 mit 215 seinen Tiefstand und im August 2026 mit 346 seinen Höchststand (typischer Sommereffekt, vgl. auch August 2025 mit 333); im September 2026 liegt er bei 252 (Januar 2025: 271). Der Abgang ist von 204 (Januar 2025) auf 293 (September 2026) gestiegen und liegt seit Mai 2026 stabil bei rund 275–300 Personen pro Monat — beides trägt zu dem in Kapitel 2 sichtbaren überwiegend negativen Netto-Saldo bei.

## 2. Netto-Fluktuation und Abgleich mit dem Bestand

Die Differenz aus Zugang und Abgang (**Netto-Fluktuation**) sollte ungefähr der monatlichen Veränderung des Arbeitslosen-Bestands entsprechen (Bestand<sub>t</sub> ≈ Bestand<sub>t-1</sub> + Zugang<sub>t</sub> − Abgang<sub>t</sub>). Ein direkter Vergleich dient als Plausibilitätscheck der Daten.

In [3]:
netto = fluktuation["Zugang (im Monat)"] - fluktuation["Abgang (im Monat)"]
bestand_delta = kennzahlen["Arbeitslose (Bestand)"].diff()

abgleich = pd.DataFrame({
    "Netto-Fluktuation (Zugang - Abgang)": netto,
    "Tatsächliche Bestandsänderung": bestand_delta,
})
abgleich["Differenz"] = abgleich["Netto-Fluktuation (Zugang - Abgang)"] - abgleich["Tatsächliche Bestandsänderung"]
abgleich.round(1)

,Netto-Fluktuation (Zugang - Abgang),Tatsächliche Bestandsänderung,Differenz
period,,,
2025-01-01,67.0,NaN,NaN
2025-02-01,-106.0,-88.0,-18.0
2025-03-01,9.0,32.0,-23.0
2025-04-01,25.0,43.0,-18.0
2025-05-01,-42.0,-19.0,-23.0
2025-06-01,52.0,71.0,-19.0
2025-07-01,-64.0,-34.0,-30.0
2025-08-01,89.0,112.0,-23.0
2025-09-01,-19.0,-2.0,-17.0


Netto-Fluktuation und tatsächliche Bestandsänderung stimmen im Vorzeichen meist überein, weichen im Betrag aber systematisch ab: Im Schnitt liegt die Netto-Fluktuation um 19,8 Personen unter der tatsächlichen Bestandsänderung (Spanne: 0 bis −30 Personen; in 19 von 20 vergleichbaren Monaten negativ, nur im September 2026 deckungsgleich). Die Bestandsveränderung fällt also durchgehend etwas positiver aus (bzw. weniger negativ), als Zugang und Abgang allein erklären. Plausibelster Grund: **Rechtskreiswechsel** zwischen SGB II und SGB III (z. B. nach Auslaufen eines Arbeitslosengeld-I-Anspruchs) verändern den SGB-II-Bestand, ohne in den Zugangs-/Abgangszahlen aus Arbeitslosigkeit insgesamt als eigene Kategorie zu erscheinen — ein Aspekt, der mit den aktuell verfügbaren Kennzahlen nicht vollständig aufgelöst werden kann.

In [4]:
fig = go.Figure()
netto_positiv = netto.where(netto >= 0)
netto_negativ = netto.where(netto < 0)
fig.add_trace(go.Bar(x=netto.index, y=netto_positiv, name="Netto-Fluktuation: Zugang > Abgang", marker_color="#dc2626"))
fig.add_trace(go.Bar(x=netto.index, y=netto_negativ, name="Netto-Fluktuation: Abgang > Zugang", marker_color="#16a34a"))
fig.add_trace(go.Scatter(
    x=bestand_delta.index, y=bestand_delta, name="Tatsächliche Bestandsänderung",
    mode="lines+markers", line=dict(color="#1e3a8a", width=2), marker=dict(size=6),
))
fig.add_hline(y=0, line_color="grey")
fig.update_layout(
    title="Netto-Fluktuation vs. tatsächliche Bestandsänderung (SGB II)",
    yaxis_title="Personen",
    xaxis_title="Monat",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.12),
    height=450,
)
export_abbildung(fig, "abb11_netto_fluktuation.png", 1000, 480)
fig

Die Grafik macht sichtbar, dass die Netto-Fluktuation (Balken) die grobe Richtung der tatsächlichen Bestandsänderung (Linie) meist mitgeht, aber fast durchgehend etwas darunter liegt — erkennbar an der Linie, die überwiegend über den Balken verläuft. Besonders im Frühjahr und Sommer 2026 (Februar–Juli, mit Ausnahme April) ist die Netto-Fluktuation negativ; nur der saisonal hohe Zugang im August 2026 (+53) sticht heraus. Das passt zu der in [`02_Analyse_SGBII.ipynb`](02_Analyse_SGBII.ipynb) beschriebenen Seitwärtsbewegung des Bestands.

## 3. 12-Monatssummen: der mittelfristige Trend

Die Monatswerte schwanken saisonal recht stark (z. B. Jahreswechsel, Ausbildungsbeginn). Die von der Bundesagentur mitgelieferte **12-Monatssumme** glättet diese Schwankungen und zeigt den mittelfristigen Trend von Zu- und Abgang.

In [5]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fluktuation.index, y=fluktuation["Zugang (12-Monatssumme)"], name="Zugang (12-Monatssumme)",
    mode="lines+markers", line=dict(color="#dc2626"), marker=dict(size=6),
))
fig.add_trace(go.Scatter(
    x=fluktuation.index, y=fluktuation["Abgang (12-Monatssumme)"], name="Abgang (12-Monatssumme)",
    mode="lines+markers", line=dict(color="#16a34a"), marker=dict(size=6),
))
fig.update_layout(
    title="Zugang und Abgang: 12-Monatssummen im Zeitverlauf (SGB II)",
    yaxis_title="Personen (rollierende 12-Monatssumme)",
    xaxis_title="Monat",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.12),
    height=450,
)
export_abbildung(fig, "abb12_12m_summen.png", 1000, 450)
fig

Der mittelfristige Trend zeigt eine sich seit Anfang 2025 öffnende Schere: Die 12-Monatssumme des Zugangs sinkt von 3.260 (Januar 2025) auf 3.182 (September 2026; Tiefpunkt 3.115 im Juni 2026), während die 12-Monatssumme des Abgangs von 3.318 auf 3.468 steigt — mit spürbarer Beschleunigung in den letzten beiden Monaten. Die daraus resultierende rollierende Netto-Fluktuation fällt entsprechend von −58 auf −286 Personen pro Jahr (bisheriger Tiefstwert) — ein deutlich negativerer Saldo als noch Anfang 2025. Trotzdem bleibt der Bestand insgesamt weitgehend stabil (siehe Kapitel 3.1 in [`02_Analyse_SGBII.ipynb`](02_Analyse_SGBII.ipynb)), was für einen gewissen Ausgleich durch andere Effekte spricht (u. a. die oben beschriebenen Rechtskreiswechsel).

## 4. Export der aufbereiteten Daten

Die Fluktuationskennzahlen werden analog zu `kennzahlen_sgb2.csv` als eigene CSV-Datei exportiert — Grundlage für den Dashboard-Tab „🔄 Fluktuation“.

In [6]:
export = fluktuation.copy()
export["Netto-Fluktuation (im Monat)"] = netto
export.to_csv(PROCESSED_DIR / "fluktuation_sgb2.csv")
export.tail()

merkmal,Abgang (12-Monatssumme),Abgang (im Monat),Zugang (12-Monatssumme),Zugang (im Monat),Netto-Fluktuation (im Monat)
period,,,,,
2026-05-01,3375.0,275.0,3238.0,243.0,-32.0
2026-06-01,3379.0,290.0,3115.0,215.0,-75.0
2026-07-01,3374.0,301.0,3146.0,273.0,-28.0
2026-08-01,3430.0,293.0,3166.0,346.0,53.0
2026-09-01,3468.0,293.0,3182.0,252.0,-41.0


## 5. Zusammenfassung

- **Zugang und Abgang** liegen im Mittel bei 270 bzw. 283 Personen pro Monat — die Fluktuation aus Arbeitslosigkeit heraus ist deutlich größer, als es der über die Zeit stabile Bestand vermuten lässt: Ständig verlassen und betreten Personen die Arbeitslosigkeit.
- **Netto-Fluktuation:** im Mittel −13,1 Personen pro Monat (13 von 21 Monaten negativ, 8 positiv) — leicht rückläufige Tendenz seit Anfang 2025.
- **Abgleich mit dem Bestand:** Die Netto-Fluktuation erklärt die tatsächliche Bestandsänderung nicht vollständig — im Schnitt eine Differenz von rund 20 Personen pro Monat, plausibel erklärbar durch Rechtskreiswechsel zwischen SGB II und SGB III.
- **12-Monatssummen:** Der Zugangstrend ist rückläufig (3.260 → 3.182 pro Jahr), der Abgangstrend steigend (3.318 → 3.468 pro Jahr) — die rollierende Netto-Fluktuation hat sich von −58 auf −286 Personen pro Jahr verschlechtert, ohne dass sich das bislang deutlich im Bestand niederschlägt.

Weiter geht es mit der Umsetzung im [Streamlit-Dashboard](../Dashboard/app.py) (Tab „🔄 Fluktuation“) sowie optional mit [`05_Demografie.ipynb`](05_Demografie.ipynb) (Altersstruktur/Geschlecht als Zeitreihe statt Momentaufnahme).